In [21]:
import os
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

In [22]:
# --- NVIDIA API Key ---
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY", "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")

In [23]:
from langchain_community.document_loaders import PyPDFLoader

# --- Load PDF with PyPDFLoader ---
loader = PyPDFLoader("cnc_troubleshooting_manual.pdf")
docs = loader.load()

print(f"Loaded {len(docs)} pages")
print(docs[0].page_content[:500])  # preview first 500 chars


Loaded 5 pages
CNC Machine Troubleshooting Manual
A practical guide to diagnosing and fixing common issues in six types of CNC machines.


In [24]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

split_docs = splitter.split_documents(docs)  # already returns Documents
docs = split_docs  # ✅ no need to wrap again


In [25]:
# --- Use NVIDIA embeddings ---
embedder = NVIDIAEmbeddings(
    model="nvidia/nv-embed-v1",
    nvidia_api_key=NVIDIA_API_KEY
)

# --- Store in FAISS ---
vdb = FAISS.from_documents(docs, embedder)
retriever = vdb.as_retriever(search_kwargs={"k": 2})
print(retriever)


tags=['FAISS', 'NVIDIAEmbeddings'] vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000219C14A4190> search_kwargs={'k': 2}


In [26]:
# --- NVIDIA LLM ---
llm = ChatNVIDIA(
    model="nvidia/llama-3.1-nemotron-70b-instruct",
    nvidia_api_key=NVIDIA_API_KEY
)


In [28]:
# --- Ask a question ---
query = "How can I fix burn marks appearing on wood while using a CNC router??"
context_docs = retriever.get_relevant_documents(query)

context = "\n\n".join([doc.page_content for doc in context_docs])
prompt = f"Answer the following based on the context:\n\n{context}\n\nQuestion: {query}"

response = llm.invoke(prompt)
print("Q:", query)
print("A:", response.content)


Q: How can I fix burn marks appearing on wood while using a CNC router??
A: Based on the provided context, to fix burn marks appearing on the material (in this case, wood) while using a **CNC Router**, follow these steps:

1. **Increase feed rate**: Try speeding up the feed rate to reduce the time the cutting tool spends on the wood, potentially minimizing the burning effect.
2. **Ensure sharp cutting tool**: A dull tool can cause excessive friction, leading to burn marks. **Sharpen or replace the cutting tool** to maintain optimal cutting conditions.
3. **Check spindle RPM**: Verify that the spindle speed (RPM) is appropriately set for the type of wood and the cutting tool in use. Adjust the **spindle RPM** as necessary to prevent overheating or excessive pressure on the material.
